# Strategy Router baseline — training, validation, your own queries

Track A of the router plan (SPEC d45/d46). Three stages, each one yours to read:

1. **Training** — fit the two logistic binaries on the decisive rows and look at which features drive each route.
2. **Validation** — score the router on held-out rows against the constant routes and the oracle ceiling.
3. **Your own queries** — route arbitrary text through the fitted model.

**Prerequisites**
- `data/route_labels/labels.parquet` and `data/feature_table/catalog.parquet` on disk.
- Stage 3 calls the taxonomy extractor, which needs spaCy: `poetry run python -m spacy download en_core_web_sm`.
- The optional full ablation downloads `multilingual-e5-small` (~470MB).

In [9]:
%load_ext autoreload
%autoreload 2

## 1 — Load the substrate

`exp.load()` joins the route labels to the 57 query features. `decisive_rows` keeps only the rows where one route clearly won (runner-up missed rank 1) — what the model trains and is scored on.

In [10]:
from IPython.display import display

from hybrid_search_rrf_dataset.router import (
    QueryEncoder,
    Representation,
    RouterExperiment,
    StrategyRouter,
    decisive_rows,
)

exp = RouterExperiment()
data = exp.load()
n_features = sum('.' in c for c in data.columns)
print(f'labelled rows: {len(data):,}  |  query features: {n_features}')

dec = decisive_rows(data)
print(f'decisive rows (trainable substrate): {len(dec):,}')
dec['winner'].value_counts().rename('decisive rows by winning route')

labelled rows: 24,338  |  query features: 57
decisive rows (trainable substrate): 2,510


winner
dense_only     1858
sparse_only     507
pure_rrf        145
Name: decisive rows by winning route, dtype: int64

## 2 — Training

Fit two one-vs-rest logistic models on the decisive rows of the training split: one for 'is this a dense win?', one for 'is this a sparse win?'. Thresholds are then tuned so ambiguous queries fall through to rrf. The coefficient tables show which features each binary leans on (weights are on standardised features, so magnitudes compare directly).

In [11]:
# Protocol (i): 80% of each lane trains, 20% is held out.
train, test = exp.split('random_within_lane')
print(f'train rows: {len(train):,}  |  held-out rows: {len(test):,}')

router = StrategyRouter(Representation.ENGINEERED).fit(train)
router.tune_thresholds(train)
print('tuned thresholds (dense, sparse):', tuple(round(t, 3) for t in router.thresholds))

train rows: 19,471  |  held-out rows: 4,867
tuned thresholds (dense, sparse): (0.15, 0.8)


In [12]:
coef = router.coefficients()
print('features pushing hardest toward DENSE:')
display(coef.sort_values('dense_weight', ascending=False).head(10).round(3))
print('features pushing hardest toward SPARSE:')
display(coef.sort_values('sparse_weight', ascending=False).head(10).round(3))

features pushing hardest toward DENSE:


,feature,dense_weight,sparse_weight
47,structured_identifiers.social_handle,0.560,-0.594
14,stopword_ratio.stopword_ratio,0.404,-0.344
55,syntactic_depth.statement_count,0.353,-0.348
41,structured_identifiers.number,0.343,-0.191
10,sentence_markers.greeting,0.333,-0.349
36,structured_identifiers.http_status_code,0.323,-0.295
12,sentence_markers.negation,0.279,-0.315
11,sentence_markers.interjection,0.246,-0.230
51,structured_identifiers.uuid,0.234,-0.244
57,derived.avg_word_length,0.207,-0.218


features pushing hardest toward SPARSE:


,feature,dense_weight,sparse_weight
6,morphology.word_variation_share,-0.541,0.548
1,length.length_chars,-0.290,0.441
8,sentence_markers.acronym,-0.113,0.259
30,structured_identifiers.error_code_like,-0.297,0.258
56,derived.identifier_density,-0.349,0.220
5,logical_structures.temporal,-0.162,0.152
25,structured_identifiers.currency_amount,-0.129,0.135
16,structured_identifiers.alt_geocoding,-0.118,0.134
3,logical_structures.math_expression,-0.073,0.102
0,coordination.widest_list_size,-0.109,0.097


## 3 — Validation

`exp.run` fits on the training portion and reports the six-column table over the held-out decisive rows, for both protocols:

- `random_within_lane` — held-out queries from lanes the model has seen.
- `holdout_lane` — the rarb-math lane held out whole (a collection never seen).

`headroom_captured` is where the router lands between the best constant route (0.0) and the oracle ceiling (1.0). The numbers are yours to read.

In [27]:
cols = [
    'protocol', 'representation', 'n_test_decisive',
    'const_dense_only', 'const_pure_rrf', 'const_sparse_only',
    'oracle', 'router', 'headroom_captured', 't_dense', 't_sparse',
]
result = exp.run(representations=[Representation.ENGINEERED])
result[cols].round(3)

holdout_lane·engineered: 100%|██████████| 2/2 [00:00<00:00, 32.16it/s, headroom=0.017, n=586]


,protocol,representation,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,497,0.738,0.225,0.222,0.973,0.751,0.057,0.15,0.80
1,holdout_lane,engineered,586,0.604,0.271,0.360,1.000,0.611,0.017,0.10,0.65


In [ ]:
# Full three-representation ablation (engineered / e5 embedding / both).
# Uncomment to run — the first call downloads multilingual-e5-small (~470MB)
# and embeds every query once (cached to data/route_labels/e5_embeddings.parquet),
# so later runs are fast.

exp_full = RouterExperiment(encoder=QueryEncoder())
runner_result= exp_full.run(autofusion=True,  autofusion_sample=50)

holdout_lane·both: 100%|██████████| 6/6 [00:00<00:00, 25.31it/s, headroom=0.008, n=586]      


In [29]:
runner_result[cols].round(3)

,protocol,representation,n_test_decisive,const_dense_only,const_pure_rrf,const_sparse_only,oracle,router,headroom_captured,t_dense,t_sparse
0,random_within_lane,engineered,497,0.738,0.225,0.222,0.973,0.751,0.057,0.15,0.80
1,random_within_lane,embedding,497,0.738,0.225,0.222,0.973,0.786,0.205,0.10,0.90
2,random_within_lane,both,497,0.738,0.225,0.222,0.973,0.787,0.211,0.15,0.90
3,holdout_lane,engineered,586,0.604,0.271,0.360,1.000,0.611,0.017,0.10,0.65
4,holdout_lane,embedding,586,0.604,0.271,0.360,1.000,0.603,-0.004,0.10,0.85
5,holdout_lane,both,586,0.604,0.271,0.360,1.000,0.607,0.008,0.10,0.85
6,random_within_lane,auto_fusion,50,0.671,0.255,0.274,0.975,0.381,-0.956,NaN,NaN
7,holdout_lane,auto_fusion,50,0.604,0.250,0.374,1.000,0.384,-0.557,NaN,NaN


## 4 — Test on your own queries

Fit on all decisive rows, then route whatever you type. This is the serving path: `predict` extracts the query's features inline and applies the same rule. Needs `en_core_web_sm` (spaCy).

In [15]:
served = StrategyRouter(Representation.ENGINEERED).fit(data)
served.tune_thresholds(data)

my_queries = [
    'what are the side effects of DHA',
    'python sort list of dicts by key',
    'CVE-2021-44228 log4j remote code execution',
    'how do mRNA vaccines work',
    'SELECT * FROM users WHERE id = 42',
]
for q in my_queries:
    print(f'{served.predict(q)!s:12s}  {q}')

dense_only    what are the side effects of DHA
dense_only    python sort list of dicts by key
sparse_only   CVE-2021-44228 log4j remote code execution
dense_only    how do mRNA vaccines work
sparse_only   SELECT * FROM users WHERE id = 42


In [16]:
# Type your own query:
served.predict('#ABBSSS')

<StrategyName.SPARSE_ONLY: 'sparse_only'>

In [20]:
served.explain('Looking for qdrant_client.http.models.FormulaQuery')

{'query': 'Looking for qdrant_client.http.models.FormulaQuery',
 'route': <StrategyName.SPARSE_ONLY: 'sparse_only'>,
 'p_dense': 0.3633248740803624,
 'p_sparse': 0.610494149608358,
 't_dense': 0.30000000000000004,
 't_sparse': 0.6,
 'dense_fires': True,
 'sparse_fires': True}

In [18]:
served.explain('http://localhost.com') # URIs have zero training coverage 

{'query': 'http://localhost.com',
 'route': <StrategyName.DENSE_ONLY: 'dense_only'>,
 'p_dense': 0.9927949789193673,
 'p_sparse': 0.00827215497082661,
 't_dense': 0.30000000000000004,
 't_sparse': 0.6,
 'dense_fires': True,
 'sparse_fires': False}

In [21]:
served.explain('qdrant_client.http.models.FormulaQuery')

{'query': 'qdrant_client.http.models.FormulaQuery',
 'route': <StrategyName.DENSE_ONLY: 'dense_only'>,
 'p_dense': 0.7343243773665759,
 'p_sparse': 0.29132406733559785,
 't_dense': 0.30000000000000004,
 't_sparse': 0.6,
 'dense_fires': True,
 'sparse_fires': False}

In [17]:
served.predict('CVE-1223')

<StrategyName.PURE_RRF: 'pure_rrf'>

In [26]:
served.predict('http://localhost.com')

<StrategyName.DENSE_ONLY: 'dense_only'>

In [18]:
from query_taxonomy.features import FeatureExtractor

fe = FeatureExtractor()

In [23]:
fe.resolve('http://localhost.com')

QueryFeatures(query_text='http://localhost.com', spans={<FeatureGroup.STRUCTURED_IDENTIFIERS: 'structured_identifiers'>: {'uri': [FeatureSpan(text='http://localhost.com', start=0, end=20)]}}, stats={<FeatureGroup.STATISTICAL_METRICS: 'statistical_metrics'>: {'length': [FeatureStat(name='length_words', value=3.0), FeatureStat(name='length_chars', value=20.0)], 'stopword_ratio': [FeatureStat(name='stopword_ratio', value=0.0)]}}, tfs={<FeatureGroup.STRUCTURED_IDENTIFIERS: 'structured_identifiers'>: {'uri': 1}})

In [22]:
from query_taxonomy.banks.tech import URIBank


uri_detector = URIBank()

uri_detector.compute("http://localhost.com")

[FeatureSpan(text='http://localhost.com', start=0, end=20)]

In [ ]:
from hybrid_search_rrf_dataset.router import _extract_features

features = _extract_features(None, "http://localhost.com")
features

{'structured_identifiers.uri': 1,
 'length.length_words': 3.0,
 'length.length_chars': 20.0,
 'stopword_ratio.stopword_ratio': 0.0,
 'natural_language_signal.natural_language_share': 0.0,
 'morphology.word_variation_share': 0.0,
 'syntactic_depth.nesting_depth': 0.0,
 'syntactic_depth.statement_count': 1.0,
 'coordination.widest_list_size': 0.0}

## 5 — How the router decides

Not a three-way classifier. Two independent one-vs-rest logistic binaries plus a hedge rule:

- `dense_fires = p_dense >= t_dense`
- `sparse_fires = p_sparse >= t_sparse`
- Neither fires → **`pure_rrf`** — the safe hedge; RRF runs both retrievers and fuses their rankings, so it cannot lose to the better of the two on top-10 metrics.
- One fires → that route.
- Both fire → whichever `p` is higher.

`pure_rrf` is never a trained class. It exists only as the residual of the hedge — the route the model picks when it is not confident enough in either alternative. The thresholds `t_dense`, `t_sparse` are learned from data by grid-searching the pair that maximizes the router's mean objective on a validation frame.

### Why `t_sparse > t_dense` is normal

The LR uses `class_weight="balanced"` on each binary. Because the decisive training set is ~74% dense / 20% sparse / 6% rrf, class balancing shifts the two binaries' output distributions apart:

- **Dense binary** — positive class dominates training. Balancing lowers its typical `p_dense`; the population sits mostly in [0.3, 0.6].
- **Sparse binary** — positive class is minority. Balancing raises its typical `p_sparse`; the population sits mostly in [0.5, 0.9].

The thresholds compensate for that shift. `t_sparse > t_dense` is **not a bias**; it is the calibration that puts the two binaries back on comparable operating points. Class imbalance is handled by `class_weight` in the loss; the thresholds handle the resulting probability-scale asymmetry at the decision layer. So `p_sparse = 0.826` and `p_dense = 0.826` do not mean the same "level of confidence" — the thresholds translate each into the right fire/hedge decision given how its binary was trained.

## 6 — What changed from the baseline

Compared to the version that first went to §3 (`router 0.678`, `headroom_captured = −0.251` on `random_within_lane` — losing to always-dense by six points), the current build lands at **`router 0.751`, `+0.057`**. Two setup-side refactors, no new data or external features.

**F2 — feature engineering.** Three derived columns computed at training and serving time (no catalog rebuild), plus one drop:

- `derived.identifier_density = sum(structured_identifiers.*) / length_words` — how much of the query is identifier-shaped.
- `derived.avg_word_length = length_chars / length_words` — the informative diff of the two length signals; long chars/word = technical vocabulary that BM25 struggles with morphologically.
- `derived.short_id_query = (identifier_density > 0) & (length_words ≤ 5)` — a boolean cross the LR cannot see linearly.
- `length.length_words` dropped after `avg_word_length` lands. In the shipped baseline it was the top sparse-puller at +1.55 — a **corpus artifact** (long BEIR / CRUMB queries happened to lean sparse), not a policy signal. Exposing the informative axis and dropping the correlated raw column dissolved the artifact.

The coefficient table in §2 shows the new picture: `derived.identifier_density` at −0.349 dense / +0.220 sparse, `derived.avg_word_length` at +0.207 dense / −0.218 sparse, `length.length_chars` still carrying part of the length signal at +0.441 sparse but no longer dominating.

**F1' — threshold tuner scope.** The tuner grid-searches `(t_dense, t_sparse)` by maximizing the router's mean objective on a validation frame. Originally that frame was "all shapes" (routes_differ + all_tied + all_zero). Measurement showed **67% of the tune frame is threshold-invariant** — all_tied rows have identical scores across route choices, all_zero rows have zero for every route. Both contribute constants to the mean regardless of threshold, diluting the argmax. The tuner was landing at `t_sparse=0.9` (sparse effectively never fires) because on the diluted objective it could not distinguish the sparse binary's genuine signal from noise.

Fix: restrict the tuner's frame to `routes_differ` rows — the ones where threshold choice actually changes the outcome. Empirical optimum shifts to `t_sparse ≈ 0.6–0.8`, sparse fires meaningfully on identifier-bearing queries, and the aggregate lift on `random_within_lane` follows.

The full amendment history — F1 was tried first as an inverse-frequency-weighted mean, sent the router into an "always rrf" degeneracy on the imbalanced argmax distribution, and was reverted as a documented negative result before F1' landed — lives in SPEC decision 47.

## 7 — Where the router still fails, and the fix path

The improved router beats the constant on aggregate, but the `explain` call above surfaces a concrete gap. For `http://localhost.com`:

```
{'p_dense': 0.993, 'p_sparse': 0.008, 'route': 'dense_only'}
```

The URL is confidently routed to dense — the wrong answer, since BM25 would trivially win on a query composed entirely of high-IDF mostly-OOV tokens.

**Root cause is coverage, not modeling.** Of the 2,510 decisive training rows, only 3 have `structured_identifiers.uri > 0`. All 3 are **long documents** (19–151 words) that happen to mention a URL — none are short URL queries. All 3 have **dense** as the winner. The LR learned "URI feature present → probably dense" from those three examples and has no counter-evidence to learn from.

More generally: the composition (SPEC decisions 31–33) was designed for **feature diversity**, which is a weak proxy for **routing-archetype coverage**. Rare-but-important archetypes (short identifiers, math expressions, entity-heavy queries) can be feature-diverse in aggregate while leaving specific `(feature × length × domain)` cells empty. Every setup-side experiment eventually hits this ceiling.